In [1]:
import pandas as pd
import io
import requests

sheet_id = "10gWTPec6n0Wbbrs4U2RUgwuxnOlYnh5gcMbKBCPlHME"
gid = "70584309"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

response = requests.get(url)
df = pd.read_csv(io.StringIO(response.text))
print(df.shape)

df.columns = ["время", "банки", "источник", "пользование",
              "опыт", "коллекторы", "плюсы_минусы",
              "осведомлен", "добавить"]
df.head()

(76, 9)


,время,банки,источник,пользование,опыт,коллекторы,плюсы_минусы,осведомлен,добавить
0,24.10.2026 13:52:22,Более 4 карт в разных банках,"Да, слышал(а) из рекламных интеграций в видео ...","Да, активно пользуюсь","Я лично использовал(а), у меня положительный опыт",NaN,NaN,Да,NaN
1,24.10.2026 13:54:35,"Пользуюсь 1 картой , в других банках счетов нет","Да, слышал(а) из рекламных интеграций в видео ...",Никогда не пользовался/ась,"Использовали мои знакомые, преимущественно нег...",NaN,"все знакомые очень тупые, не умеют распоряжать...",Нет,нит
2,24.10.2026 13:56:01,Более 4 карт в разных банках,"Да, слышал(а) из рекламных интеграций в видео ...",Никогда не пользовался/ась,"Использовали мои знакомые, преимущественно нег...",NaN,процент растущий,Нет,-
3,24.10.2026 13:57:23,"Пользуюсь 1 картой , в других банках счетов нет","Да, слышал(а) от друзей/родственников","Да, но в процессе закрытия этой карты (не трач...","Использовали мои знакомые, преимущественно нег...",NaN,NaN,Нет,NaN
4,24.10.2026 13:57:44,Пользуюсь 2 картами из разных банков,"Да, слышал(а) из рекламных интеграций в видео ...","УЖЕ не пользуюсь, раньше пользовался/ась","Я лично использовал(а), у меня негативный опыт",NaN,столкнулся с собственной финансовой неграмотно...,Да,NaN


In [2]:
#это анализ поведения клиентов в нашем сегменте (18-29)

In [3]:
df['банки'] = df['банки'].replace("Более 4 карт в разных банках", "4 и более карт в разных банках")
df['банки'].value_counts()

банки
Пользуюсь 2 картами из разных банков               27
4 и более карт в разных банках                     18
Пользуюсь 3 картами из разных банков               16
Пользуюсь 1 картой , в других банках счетов нет    15
Name: count, dtype: int64

In [4]:
df['осв'] = df['осведомлен'].map({'Да' : 1, 'Нет' : 0 })
df['осв'].value_counts()

осв
1.0    29
0.0    17
Name: count, dtype: int64

In [5]:
df['лично'] = df['опыт'].str.startswith('Я лично').astype(int)
df['лично'].value_counts()
df['негатив'] = df['опыт'].map({
    'Я лично использовал(а), у меня негативный опыт' : 1,
    'Использовали мои знакомые, преимущественно негативный опыт' : 1,
    'Я лично использовал(а), у меня положительный опыт' : 0,
    'Использовали мои знакомые, преимущественно положительный опыт' : 0

})
df['осв'] = df['осведомлен'].map({'Да': 1, 'Нет': 0})
df.groupby('осведомлен')['негатив'].agg(['mean','count'])

# тут по процентам mean = процент от count(всего)
# например каунт 20 - меан 0.5 - значит 10/20 человек кто осведомлен имели негативный опыт

,mean,count
осведомлен,,
Да,0.500000,28
Нет,0.875000,16
Нет опыта,0.615385,13


 щас проверим насколько это случайно

### н0 = разницы нет (доля негатива у знавших и не знавших условия одинаковая - осведомлённость с негативом не связана, а разница в данных появилась случайно )
### н1 = есть разница (осведомлённость связана с негативным опытом)

In [6]:
from scipy import *

know = df.loc[df['осв'] == 1,'негатив'].dropna()
dont_know = df.loc[df['осв'] == 0,'негатив'].dropna()

res = stats.ttest_ind(dont_know, know,equal_var=False)
print(f't стат:{res.statistic:.2f}')
print(f'пи валью:{res.pvalue:.4f}')

if res.pvalue < 0.05:
    print('отвергаем н0, что разницы нет - разница есть')
else:
    print('разницы нет')

a = 0.05
t_crit = stats.t.ppf(1 - a / 2, res.df)
print(f'степени свободы:{res.df:.1f}')
print(f'т крит:{t_crit:.2f}')


if abs(res.statistic) > t_crit:
    print('тест стьюдента показал что он больше критического - значит разброс не случайный и осведомленность связана с негативным опытом')
else:
    print('разброс случайный и значит осведомленность не влияет на негативный опыт')

import statsmodels.formula.api as smf
m = smf.ols('негатив ~ осв + лично', data=df).fit(cov_type='HC1')
print(m.summary().tables[1])

    # тут док-во просто следи за выводом

t стат:2.91
пи валью:0.0058
отвергаем н0, что разницы нет - разница есть
степени свободы:40.8
т крит:2.02
тест стьюдента показал что он больше критического - значит разброс не случайный и осведомленность связана с негативным опытом
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9413      0.086     11.008      0.000       0.774       1.109
осв           -0.2708      0.144     -1.878      0.060      -0.554       0.012
лично         -0.2652      0.152     -1.746      0.081      -0.563       0.032


щас я проверю наскок верна моя теория что брали карты не глядя

In [7]:
df['юз_личный'] = df['пользование'].str.startswith(('Да', 'УЖЕ')).astype(int)

use_lose = df.loc[(df['юз_личный'] == 1) & (df['осв'] == 0),['пользование', 'осв', 'опыт']]

total = ((df['юз_личный'] == 1) & (df['осв'].notna())).sum()
print(total)
print(f'пользовались и не знали условий:{len(use_lose)} из {total} = {len(use_lose)/ total:.1%}')


24
пользовались и не знали условий:8 из 24 = 33.3%


получается что те кто брали не глядя стремится к 1/3 от всех, кто когда-либо пользовался

теперь посмотрим связано ли личное пользование с рекомендацией из инета или рекомендация друзей - важно для канала продаж

In [8]:
df.groupby("пользование").agg({
    "негатив": ["mean", "count"],
    "осв": "mean",
})

негатив             осв
                                                        mean count      mean
пользование                                                                 
Да, активно пользуюсь                               0.250000     4  0.750000
Да, но в процессе закрытия этой карты (не трачу...  0.571429     7  0.571429
Да, но не пользуюсь активно (трачу деньги с это...  0.375000     8  0.750000
Никогда не пользовался/ась                          0.696970    33  0.590909
УЖЕ не пользуюсь, раньше пользовался/ась            1.000000     5  0.600000

щас сделаем сегментацию

In [9]:
df['сегмент'] = df['пользование'].map({
    'Да, активно пользуюсь' : 'пользуются',
    "Да, но не пользуюсь активно (трачу деньги с этой карты)": "пользуются",
    "Да, но в процессе закрытия этой карты (не трачу, только выплачиваю ее )": "уходят",
    "УЖЕ не пользуюсь, раньше пользовался/ась": "уходят",
    "Никогда не пользовался/ась": "никогда",

})
# всего 3 категории кто пользуется кто уходит и кто никогда не трогал
print(df['сегмент'].value_counts())
df.groupby('сегмент').agg({
    'негатив':['count'],
    'осв':['mean'],
}).round(2)

res1 = df.groupby('сегмент')['негатив'].sum()

total_in = df.groupby('сегмент')['негатив'].mean()
negative_in = df.groupby('сегмент')['негатив'].sum()
res = (df.groupby('сегмент')['негатив'].mean() * 100).round(1)
print(f'проценты негатива по сегментам')
print(res)

сегмент
никогда       52
пользуются    12
уходят        12
Name: count, dtype: int64
проценты негатива по сегментам
сегмент
никогда       69.7
пользуются    33.3
уходят        75.0
Name: негатив, dtype: float64
